# B7 Stage1 — implementation release, formal run disabled

Conditions Freeze: `5dbac7af2d41aa912308868e6ad1a6dbf7cdd107`. Pure Time primary + five fixed SL robustness; 90 jobs, 7,335,360 structures, 44,012,160 variants. No U06+, Validation or Monitor execution.

Use only after reviewing the Implementation Freeze in Chat. Supply its exact SHA below. All switches default false. Enable formal execution only after a separate Chat approval. Candidate output is reviewable only at COMPLETE_STAGE1_ONLY. No intermediate candidate names/times/results are printed.

In [ ]:
from pathlib import Path
RUN_MOUNT = False
RUN_INSTALL = False
RUN_CHECKOUT = False
RUN_INPUT_AUDIT = False
RUN_TESTS = False
RUN_BOUNDED_SMOKE = False
RUN_PREFLIGHT = False
RUN_FORMAL = False
RUN_RESUME = False
RUN_ARCHIVE = False
IMPLEMENTATION_SHA = "PASTE_REVIEWED_IMPLEMENTATION_FREEZE_SHA"
CHAT_APPROVAL = ""  # Only after separate Chat approval: CHAT_APPROVED_COLAB_STAGE1_ONLY
REPO = Path("/content/time-entry-portfolio-lab")
DATA_ROOT = Path("/content/drive/MyDrive/SET_EXACT_FROZEN_COLLECTION_ROOT")
OUTPUT = Path("/content/b7_stage1")
DRIVE_ARCHIVE = Path("/content/drive/MyDrive/SET_NEW_B7_STAGE1_RUN_FOLDER")


## Mount / environment
Mounting does not start research. Pin NumPy/pandas, restart the Colab runtime after installation, then verify the actual Python version. Python differences require exact reference replay. Never silently continue with mismatched dependencies.

In [ ]:
if RUN_MOUNT:
    from google.colab import drive
    drive.mount("/content/drive")
if RUN_INSTALL:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy==2.3.5", "pandas==2.2.3"])
    raise RuntimeError("Restart runtime after dependency installation, then re-run setup with RUN_INSTALL=False")


## Reviewed implementation checkout
No checkout overwrite. This cell creates a new clone, then checks out the reviewed immutable commit. Authentication, if required, must use your normal GitHub setup; do not paste credentials into this notebook.

In [ ]:
if RUN_CHECKOUT:
    import subprocess
    if len(IMPLEMENTATION_SHA) != 40 or not all(c in "0123456789abcdef" for c in IMPLEMENTATION_SHA):
        raise ValueError("Set the reviewed implementation SHA")
    if REPO.exists():
        raise FileExistsError("Use a fresh directory; do not overwrite an existing checkout")
    subprocess.check_call(["git", "clone", "https://github.com/TR-KJ/time-entry-portfolio-lab.git", str(REPO)])
    subprocess.check_call(["git", "-C", str(REPO), "checkout", "--detach", IMPLEMENTATION_SHA])


## Input audit, tests, smoke and full preflight
The collection resolver accepts only the exact 72 manifest filenames. It does not choose alternative exports. Input audit checks filename/hash/rows/raw endpoints; research arrays are copied to Discovery only. Smoke is fixed at 2020-02-04 09:00 JST, 30-minute hold, nine pairs/both directions/Pure + five SL; only replay status is reported.

In [ ]:
if any((RUN_INPUT_AUDIT, RUN_TESTS, RUN_BOUNDED_SMOKE, RUN_PREFLIGHT, RUN_FORMAL, RUN_RESUME, RUN_ARCHIVE)):
    import sys
    sys.dont_write_bytecode = True  # Keep the reviewed checkout clean during imports.
    sys.path.insert(0, str(REPO / "src/research"))
    from b7.stage1_input import resolve_exact, audit_inputs
    from b7.stage1_smoke import run_smoke
    from b7.stage1_runtime import environment, run_tests, preflight, run_formal, archive_completed
    print("Environment:", environment())
    if any((RUN_INPUT_AUDIT, RUN_BOUNDED_SMOKE, RUN_PREFLIGHT, RUN_FORMAL, RUN_RESUME)):
        paths = resolve_exact(DATA_ROOT)
if RUN_INPUT_AUDIT:
    audit = audit_inputs(paths)
    print("Exact files verified:", len(audit))
if RUN_TESTS:
    print(run_tests()["Status"])
if RUN_BOUNDED_SMOKE:
    smoke = run_smoke(paths)
    print("Bounded exact replay:", smoke["Status"], smoke["CaseCount"], "cases")
if RUN_PREFLIGHT:
    # Use separate preflight output so a later fresh formal folder does not already exist.
    proof = preflight(paths, IMPLEMENTATION_SHA, Path("/content/b7_stage1_preflight"))
    print(proof["Status"])


## Formal / resume — OFF by default
Formal execution repeats all barriers even when preflight was run earlier. Resume accepts only exact identities and output hashes. Incomplete jobs are recomputed from scratch; completed jobs alone are reusable. RUN_RESUME has no effect without RUN_FORMAL. The current implementation request does not authorize this cell to run.

In [ ]:
if RUN_FORMAL:
    review = run_formal(paths, IMPLEMENTATION_SHA, OUTPUT, approval=CHAT_APPROVAL,
                        resume=RUN_RESUME, progress=lambda p: print(p))
    print(review["Status"])


## Archive after COMPLETE only
Choose a new result-independent destination, for example `b7_stage1_<date-or-run-id>`. Existing folders are rejected. Only manifested outputs, review records, completion markers and archive ZIP are saved; raw M1 is never copied.

In [ ]:
if RUN_ARCHIVE:
    print(archive_completed(OUTPUT, DRIVE_ARCHIVE))
